# Agent: Observer Agent (Sensor Network)

**Purpose:** Deploy water-level sensors and publish readings every 2 seconds.

**Sensors:** 5 sensors deployed around Køge Søndre Strand (55.4506, 12.1975)

**Publishes to:** `city/flood/observer/sensor_1` through `sensor_5` topics as JSON `ObserverReading` messages

**Data:** Water level (meters) + flow rate (simulated)

In [ ]:
import time
import json
import random
from datetime import datetime, timezone
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, make_topic
from simulated_city.flood import ObserverReading

# Load configuration
try:
    cfg = load_config()
    print(f"MQTT Broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
    print(f"Base Topic: {cfg.mqtt.base_topic}")
except Exception as e:
    print(f"Error loading config: {e}")
    cfg = None

# Define sensor locations (around Køge Søndre Strand)
BASE_LAT, BASE_LNG = 55.4506, 12.1975
num_sensors = 5
sensors = {
    f"sensor_{i+1}": (BASE_LAT + random.uniform(-0.0005, 0.0005), 
                      BASE_LNG + random.uniform(-0.0005, 0.0005))
    for i in range(num_sensors)
}

# Flood cycle configuration
BASELINE_WATER = 0.2
RISE_RATE = 0.20
RISE_DURATION_S = 30.0
FALL_DURATION_S = 15.0
CYCLE_DURATION_S = RISE_DURATION_S + FALL_DURATION_S
MAX_FLOOD_WATER = BASELINE_WATER + (RISE_RATE * RISE_DURATION_S)  # 6.2m

# Global cycle state
global_water_level = BASELINE_WATER
cycle_elapsed_s = 0.0
current_phase = "rising"

print("\nObserver network configured:")
print(f"  Base location: Køge Søndre Strand ({BASE_LAT}, {BASE_LNG})")
print(f"  Sensors: {num_sensors}")
print(f"  Cycle: rise {RISE_DURATION_S:.0f}s at {RISE_RATE:.2f}m/s, fall {FALL_DURATION_S:.0f}s")
for sensor_id, (lat, lng) in sensors.items():
    print(f"    - {sensor_id}: ({lat:.4f}, {lng:.4f})")

MQTT Broker: 127.0.0.1:1883
Base Topic: simulated-city

Observer network configured:
  Base location: Køge Søndre Strand (55.4506, 12.1975)
  Sensors: 5
    - sensor_1: (55.4504, 12.1972)
    - sensor_2: (55.4510, 12.1972)
    - sensor_3: (55.4507, 12.1977)
    - sensor_4: (55.4509, 12.1979)
    - sensor_5: (55.4503, 12.1976)


In [ ]:
if cfg:
    mqtt = MqttConnector(cfg.mqtt, client_id_suffix='observer')
    mqtt.connect()
    if mqtt.wait_for_connection(timeout=5):
        print("\n✓ Connected to MQTT broker")
    else:
        print("✗ Failed to connect to MQTT broker")
        mqtt = None


✓ Connected to MQTT broker


In [ ]:
def update_global_water_level(elapsed_s: float) -> float:
    """Update water level on repeating cycle: rise 30s, fall 15s."""
    global global_water_level, cycle_elapsed_s, current_phase

    cycle_position = cycle_elapsed_s % CYCLE_DURATION_S
    
    if cycle_position < RISE_DURATION_S:
        # Rising phase: increase by RISE_RATE per second
        fraction = cycle_position / RISE_DURATION_S
        global_water_level = BASELINE_WATER + (RISE_RATE * cycle_position)
        current_phase = "rising"
    else:
        # Falling phase: decrease back to baseline
        fall_position = cycle_position - RISE_DURATION_S
        fraction = fall_position / FALL_DURATION_S
        global_water_level = MAX_FLOOD_WATER - (RISE_RATE * fall_position)
        current_phase = "falling"
    
    global_water_level = max(BASELINE_WATER, min(MAX_FLOOD_WATER, global_water_level))
    cycle_elapsed_s += elapsed_s
    return global_water_level


def publish_sensor_reading(sensor_id: str, lat: float, lng: float):
    """Publish a sensor reading."""
    water_level = global_water_level + random.uniform(-0.05, 0.05)  # ±50mm noise
    flow_rate = max(0, water_level - BASELINE_WATER) * 2.0  # Proportional to height
    
    reading = ObserverReading(
        sensor_id=sensor_id,
        water_level=max(0, water_level),
        flow_rate=flow_rate,
        timestamp=datetime.now(timezone.utc).isoformat()
    )
    
    topic = make_topic(cfg.mqtt, "flood", "observer", sensor_id)
    mqtt.publish_json_checked(topic, reading.to_json(), description=f"{sensor_id}: {water_level:.2f}m")
    return reading


# Main simulation loop
try:
    print("\n📊 Starting observer network (publishing every 2 seconds)...")
    print(f"Topics: city/flood/observer/sensor_*")
    print("Press Ctrl+C to stop\n")
    
    tick = 0
    last_time = time.time()
    
    while True:
        now = time.time()
        elapsed = now - last_time
        last_time = now
        
        # Update cycle
        update_global_water_level(elapsed)
        
        # Publish every 2 seconds
        if tick % 2 == 0:
            print(f"[{datetime.now(timezone.utc).strftime('%H:%M:%S')}] Phase: {current_phase:7s} | Water: {global_water_level:.2f}m")
            for sensor_id, (lat, lng) in sensors.items():
                publish_sensor_reading(sensor_id, lat, lng)
        
        time.sleep(1)
        tick += 1
        
except KeyboardInterrupt:
    print("\n\nShutdown observer agent")
    if mqtt:
        mqtt.disconnect()


In [ ]:
print("Starting observer network loop...")
print("Polling interval: 2s")
print("Press Ctrl+C to stop\n")

reading_count = 0
try:
    while True:
        reading_count += 1
        update_global_water_level(2.0)

        readings_str = ""
        for sensor_id in sorted(sensors.keys()):
            publish_sensor_reading(sensor_id)
            level = get_sensor_reading(sensor_id)
            readings_str += f"  {sensor_id}: {level:.2f}m"

        print(f"[{reading_count}] Phase: {current_phase} | Base water: {global_water_level:.2f}m |{readings_str}")
        time.sleep(2)

except KeyboardInterrupt:
    print("\n\n✓ Observer agent stopped by user")
finally:
    if mqtt:
        mqtt.disconnect()

Starting observer network loop...
Polling interval: 5s
Press Ctrl+C to stop

[1]   sensor_1: 0.17m  sensor_2: 0.21m  sensor_3: 0.20m  sensor_4: 0.21m  sensor_5: 0.17m
[2]   sensor_1: 0.16m  sensor_2: 0.21m  sensor_3: 0.25m  sensor_4: 0.22m  sensor_5: 0.18m
[3]   sensor_1: 0.16m  sensor_2: 0.20m  sensor_3: 0.19m  sensor_4: 0.16m  sensor_5: 0.22m
[4]   sensor_1: 0.24m  sensor_2: 0.24m  sensor_3: 0.15m  sensor_4: 0.25m  sensor_5: 0.19m
[5]   sensor_1: 0.16m  sensor_2: 0.16m  sensor_3: 0.23m  sensor_4: 0.21m  sensor_5: 0.23m
[6]   sensor_1: 0.17m  sensor_2: 0.18m  sensor_3: 0.25m  sensor_4: 0.15m  sensor_5: 0.19m
[7]   sensor_1: 0.18m  sensor_2: 0.23m  sensor_3: 0.24m  sensor_4: 0.25m  sensor_5: 0.15m
[8]   sensor_1: 0.18m  sensor_2: 0.22m  sensor_3: 0.17m  sensor_4: 0.20m  sensor_5: 0.21m
[9]   sensor_1: 0.19m  sensor_2: 0.24m  sensor_3: 0.20m  sensor_4: 0.23m  sensor_5: 0.16m
[10]   sensor_1: 0.23m  sensor_2: 0.25m  sensor_3: 0.16m  sensor_4: 0.20m  sensor_5: 0.17m
[11]   sensor_1: 0.15m

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...




✓ Observer agent stopped by user


In [ ]:
print("Starting observer network loop...")
print("Polling interval: 2s")
print("Press Ctrl+C to stop\n")

reading_count = 0
try:
    while True:
        reading_count += 1
        update_global_water_level(2.0)

        readings_str = ""
        for sensor_id in sorted(sensors.keys()):
            publish_sensor_reading(sensor_id)
            level = get_sensor_reading(sensor_id)
            readings_str += f"  {sensor_id}: {level:.2f}m"

        print(f"[{reading_count}] Phase: {current_phase} | Base water: {global_water_level:.2f}m |{readings_str}")
        time.sleep(2)

except KeyboardInterrupt:
    print("\n\n✓ Observer agent stopped by user")
finally:
    if mqtt:
        mqtt.disconnect()

Starting observer network loop...
Polling interval: 5s



NameError: name 'connector' is not defined

## Sensor Network Loop

Each sensor publishes readings every 5 seconds. All sensors measure the same water level
(representing a unified water body), but with independent sensor noise.

Run this cell to start the observer network (press Ctrl+C to stop).

In [12]:
# Shared water level state (simulating a physical water body)
# In reality, all sensors would measure the same water level
water_level_state = {"level": 0.0, "lock": threading.Lock()}

def get_current_water_level():
    """Get the current water level (shared across all sensors)."""
    with water_level_state["lock"]:
        return water_level_state["level"]

def set_water_level(level):
    """Set the water level (called by external trigger simulation)."""
    with water_level_state["lock"]:
        water_level_state["level"] = max(0, level)

def get_iso_timestamp():
    """Return current time as ISO 8601 string."""
    return datetime.now(timezone.utc).isoformat()

def publish_reading(sensor_id, water_level, flow_rate=0.0):
    """Publish a water level reading from a sensor."""
    # Add small sensor noise
    noisy_level = water_level + random.uniform(-0.2, 0.2)
    noisy_level = max(0, noisy_level)  # Water level can't be negative
    
    reading = ObserverReading(
        sensor_id=sensor_id,
        water_level=noisy_level,
        flow_rate=flow_rate,
        timestamp=get_iso_timestamp()
    )
    
    topic = make_topic(mqtt_cfg, "observer", sensor_id)
    payload = json.dumps(reading.to_json())
    publisher.publish_json(topic, payload, qos=1)
    
    return reading

print("Water level simulation configured.")

Water level simulation configured.


## Water Level Simulation

Model simulates a deterministic flood cycle:
- Baseline starts at 0.20m
- Rising phase: +0.20m per second for 30 seconds
- Peak level: 6.20m
- Falling phase: returns to 0.20m over 15 seconds
- Cycle repeats continuously

Each sensor adds small noise ±0.05m to simulate realistic variations.

In [11]:
# Connect to MQTT
connector = MqttConnector(mqtt_cfg, client_id_suffix="observer")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

✓ Connected to MQTT broker


## Connect to MQTT Broker

In [5]:
import time
import json
import random
from datetime import datetime, timezone
import threading

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import ObserverReading

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

# Sensor network configuration
KOEGE_STRAND_LAT = 55.45058843620187
KOEGE_STRAND_LON = 12.197503222443261
NUM_SENSORS = 5
SENSOR_SPACING_KM = 0.002  # ~200 meters spacing

# Create sensor locations (small grid around Køge Søndre Strand)
SENSOR_LOCATIONS = []
for i in range(NUM_SENSORS):
    lat = KOEGE_STRAND_LAT + (i % 3) * SENSOR_SPACING_KM / 111  # 111 km per degree
    lon = KOEGE_STRAND_LON + (i // 3) * SENSOR_SPACING_KM / 111
    SENSOR_LOCATIONS.append({
        "id": f"sensor_{i+1}",
        "lat": lat,
        "lon": lon
    })

print(f"Observer network configured:")
print(f"  Base location: Køge Søndre Strand ({KOEGE_STRAND_LAT}, {KOEGE_STRAND_LON})")
print(f"  Sensors: {NUM_SENSORS}")
for loc in SENSOR_LOCATIONS:
    print(f"    - {loc['id']}: ({loc['lat']:.4f}, {loc['lon']:.4f})")

Observer network configured:
  Base location: Køge Søndre Strand (55.45058843620187, 12.197503222443261)
  Sensors: 5
    - sensor_1: (55.4506, 12.1975)
    - sensor_2: (55.4506, 12.1975)
    - sensor_3: (55.4506, 12.1975)
    - sensor_4: (55.4506, 12.1975)
    - sensor_5: (55.4506, 12.1975)


## Setup and Configuration

# Agent: Observer (Water Level Sensors)

Sensor network that monitors water levels near Køge Søndre Strand (beach).

**Setup:**
- 5 sensors deployed near Køge Søndre Strand
- Each publishes water level readings every 2 seconds
- Water follows cycle: rise at 0.20m/s for 30s, fall to 0.20m in 15s
- Simulates realistic sensor variability with small noise (±0.05m)

**Coordinates:** All sensors centered around 55.4506, 12.1975 (Køge Søndre Strand)

# Agent Observer
Sensor notebook that reads water levels and publishes `ObserverReading` messages.
Can be instantiated multiple times with different IDs.